# Hypnos reference notebook: the population-variability layer (v0.2)

**NOT FOR CLINICAL USE.** Research / education / simulation only.

This notebook is executed in CI (via `nbmake`) so it cannot rot. It reproduces the v0.2
headline: seeded between-subject-variability prediction bands, the *never-synthesize* rule,
the uncertainty-aware divergence readout (variance decomposition + separation index), and the
PD effect band. Every number is a deterministic projection of the curated dataset — nothing is
invented. See [`docs/specs/v0.2/variability.md`](../docs/specs/v0.2/variability.md).

In [ ]:
import numpy as np
import hypnos

ds = hypnos.load()
assert hypnos.validate_dataset(ds) == [], 'dataset must be valid'

# Which models carry a curated random-effects (Ω/Σ) layer, and which honestly carry none?
with_bsv = [m.id for m in ds if m.has_published_variability]
without_bsv = [m.id for m in ds if m.purpose == 'pk' and not m.has_published_variability]
print('curated variability:', with_bsv)
print('no published BSV   :', [i.split('.')[-1] for i in without_bsv])

## Seeded prediction bands and the never-synthesize rule

Only Eleveld 2018 publishes the random-effects structure, so only it earns a 5–95% prediction
band (`variability_status: diagonal`). Marsh and Schnider publish no between-subject variability,
so Hypnos draws **no band** for them — it will not borrow a sibling's Ω or impute a CV. *A missing
band is a true statement; a borrowed one is a lie with error bars.*

In [ ]:
patient = dict(age=72, weight=60, height=162, sex='F')
schedule = [('bolus', 0.0, '2 mg/kg'), ('infusion', 0.0, '6 mg/kg/h')]
t = np.linspace(0, 60, 361)

elev = hypnos.simulate(ds, 'hypnotics_iv.propofol.eleveld_2018', patient=patient,
                       schedule=schedule, t=t, bands=True, percentile=(5, 95),
                       samples=2000, seed=7)
i = int(np.argmax(elev.ce_quantiles[50]))   # the median-peak instant
print(f"Eleveld  variability_status={ds['hypnotics_iv.propofol.eleveld_2018'].variability_status}"
      f"  band_tier={elev.band_tier}")
print(f"  Ce band @ peak: {elev.ce_quantiles[50][i]:.2f} "
      f"[{elev.ce_quantiles[5][i]:.2f}, {elev.ce_quantiles[95][i]:.2f}] ug/mL (5-95%, seeded)")

marsh = hypnos.simulate(ds, 'hypnotics_iv.propofol.marsh_1991', patient=patient,
                        schedule=schedule, t=t, bands=True, seed=7)
assert elev.ce_quantiles is not None, 'Eleveld must carry a band'
assert marsh.ce_quantiles is None, 'no-BSV model must draw no band (never-synthesize)'
print('\nMarsh band:', marsh.ce_quantiles, '—', next(w for w in marsh.warnings if w.startswith('BAND')))

## Determinism is mandatory

Every band-producing call takes an explicit integer seed; identical `(seed, dataset_version,
request)` yields byte-identical quantiles (spec §6). No band is ever drawn from an unseeded
generator — `compare(bands=True)` without a seed raises.

In [ ]:
again = hypnos.simulate(ds, 'hypnotics_iv.propofol.eleveld_2018', patient=patient,
                        schedule=schedule, t=t, bands=True, percentile=(5, 95),
                        samples=2000, seed=7)
for q in (5, 50, 95):
    assert np.array_equal(elev.ce_quantiles[q], again.ce_quantiles[q])
print('reproducible: same seed -> byte-identical 5/50/95 quantiles')

try:
    hypnos.compare(ds, drug='propofol', patient=patient, schedule=schedule, t=t, bands=True)
    raise AssertionError('unseeded bands must be refused')
except ValueError as e:
    print('unseeded band refused:', e)

## Uncertainty-aware divergence: what dominates the uncertainty here?

The total predictive variance decomposes into **structural** (between-model), **BSV** (within-model,
η/Ω), and **residual** (Σ) components, time-resolved. For this elderly patient the band-eligible set
is Eleveld alone, so the structural share is 0 and the within-model variance is dominated by
between-subject scatter — *the patient is the uncertainty here, not the assay*. Models with no
published BSV are **named**, never silently dropped.

In [ ]:
cmp = hypnos.compare(ds, drug='propofol', patient=patient, schedule=schedule, t=t,
                     bands=True, percentile=(5, 95), samples=2000, seed=7)
vs = cmp.divergence['ce']['variance_share']
print(f"variance share @ t*={vs['t_star_min']:.2f} min: "
      f"structural {vs['structural']:.2f} | BSV {vs['bsv']:.2f} | residual {vs['residual']:.2f}")
assert abs(vs['structural'] + vs['bsv'] + vs['residual'] - 1.0) < 1e-6
for e in cmp.excluded_from_bands:
    print('excluded from band metrics:', e['model_id'].split('.')[-1], '—', e['reason'])
# The separation index needs >= 2 band-eligible models; today only Eleveld qualifies for propofol,
# so it is honestly absent rather than fabricated from a single band.
print('separation present:', 'separation' in cmp.divergence['ce'])

## The effect band: PK variability propagated into BIS space

Compose the band-eligible PK model with a PD model and the same Ω that scatters the concentration
scatters the predicted effect: each virtual individual's true effect-site curve is pushed through
the (deterministic) Hill link, and quantiles are taken on the *effect* draws directly (correct
under the monotone, non-linear transform). Because PD-parameter BSV (Ce50, γ) is **not** curated,
the effect band is an honest **lower bound** on true effect spread.

In [ ]:
eff = hypnos.simulate(ds, 'hypnotics_iv.propofol.eleveld_2018', patient=patient,
                      schedule=schedule, t=t, pd_model='pd_effect.propofol.eleveld_bis',
                      bands=True, percentile=(5, 95), samples=2000, seed=7)
j = int(np.argmin(eff.effect_quantiles[50]))   # peak effect = minimum median BIS
print(f"effect band @ peak BIS: {eff.effect_quantiles[50][j]:.1f} "
      f"[{eff.effect_quantiles[5][j]:.1f}, {eff.effect_quantiles[95][j]:.1f}] (5-95%, PK-BSV only)")
assert eff.effect_quantiles[5][j] < eff.effect_quantiles[95][j]
print('caveat:', next(w for w in eff.warnings if 'LOWER BOUND' in w))

In [ ]:
# Optional plot (skipped if matplotlib is absent; the curated figures live in docs/images/).
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.fill_between(t, elev.ce_quantiles[5], elev.ce_quantiles[95], alpha=0.18,
                    label='Eleveld 5-95% band')
    ax.plot(t, elev.ce_quantiles[50], lw=2, label='Eleveld median')
    ax.plot(t, marsh.ce, '--', lw=1.5, label='Marsh (no band — never-synthesize)')
    ax.set_xlabel('time (min)'); ax.set_ylabel('effect-site conc (ug/mL)')
    ax.set_title('Hypnos v0.2 — seeded prediction band (NOT FOR CLINICAL USE)')
    ax.legend()
except ImportError:
    print('matplotlib not installed; skipping plot')